# PRism reviewer fine-tuning (Kaggle)

QLoRA fine-tunes Qwen2.5-Coder-3B-Instruct with Unsloth on CodeReviewer comment generation,
compares base and tuned on 500 held-out hunks and exports a GGUF for Ollama. All logic lives in
`training/sft/` in the repo; this notebook only runs it. Settings: Accelerator GPU T4 (one GPU is
used), Internet on. Details in `training/README.md`.

## 1. Settings

`TRAIN_ROWS` is a seeded sample of the cleaned train split. Everything the run writes goes straight
to `OUT` under `/kaggle/working`, which Kaggle keeps even if the session is cut off.

In [ ]:
REPO = "ManvGit17/prism"
REF = "phase-4"  # branch, tag or commit
TRAIN_ROWS = 40000
WORK = "/kaggle/temp"  # dataset and dry run; not saved
OUT = "/kaggle/working/sft_out"

## 2. Secrets

Reads the secrets added under Add-ons, Secrets. `GITHUB_TOKEN` is needed to clone the private
repo; `WANDB_API_KEY` and `HF_TOKEN` are optional. Only yes or no is printed, never a value.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

_client = UserSecretsClient()


def secret(name):
    try:
        return _client.get_secret(name) or None
    except Exception:
        return None


GITHUB_TOKEN = secret("GITHUB_TOKEN")
for name in ("WANDB_API_KEY", "HF_TOKEN"):
    value = secret(name)
    if value:
        os.environ[name] = value
print("GITHUB_TOKEN set:", bool(GITHUB_TOKEN))
print("WANDB_API_KEY set:", "WANDB_API_KEY" in os.environ)
print("HF_TOKEN set:", "HF_TOKEN" in os.environ)
assert GITHUB_TOKEN, (
    "Add a GITHUB_TOKEN secret (read access to the repo) and attach it to this notebook."
)

## 3. Clone the repo

The token goes to git as a request header through environment variables. It is not in the URL, not
in `.git/config`, and any git output is scrubbed before it is printed.

In [ ]:
import base64
import subprocess

_basic = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
_env = dict(
    os.environ,
    GIT_CONFIG_COUNT="1",
    GIT_CONFIG_KEY_0="http.extraheader",
    GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}",
    GIT_TERMINAL_PROMPT="0",
)


def scrub(text):
    return text.replace(GITHUB_TOKEN, "***").replace(_basic, "***")


def git(*args, cwd=None):
    proc = subprocess.run(["git", *args], cwd=cwd, env=_env, capture_output=True, text=True)
    print(scrub(proc.stdout + proc.stderr).strip())
    if proc.returncode:
        raise SystemExit(f"git {args[0]} failed")


os.chdir("/kaggle/working")
if not os.path.isdir("prism"):
    git("clone", "--quiet", f"https://github.com/{REPO}.git", "prism")
os.chdir("/kaggle/working/prism")
git("fetch", "--quiet", "origin")
git("checkout", "--quiet", REF)
# On a branch, move to the latest pushed commit: a session that cloned earlier would be stale.
_remote = subprocess.run(
    ["git", "rev-parse", "--verify", "--quiet", f"origin/{REF}"], capture_output=True
)
if _remote.returncode == 0:
    git("reset", "--hard", "--quiet", f"origin/{REF}")
git("log", "-1", "--oneline")

## 4. Install requirements

Unsloth brings its own pinned peft, trl and bitsandbytes. The second line adds what the PRism code
imports.

In [ ]:
!pip install -q unsloth
!pip install -q -r training/requirements-kaggle-sft.txt
import torch

print(
    "torch",
    torch.__version__,
    "cuda:",
    torch.cuda.is_available(),
    torch.cuda.device_count(),
    "GPUs",
)

## 5. Download the dataset

Gets `Comment_Generation.zip` (847 MB) from Zenodo. It is read in place, never unpacked.

In [ ]:
import zipfile

DATA = f"{WORK}/Comment_Generation.zip"
URL = "https://zenodo.org/records/6900648/files/Comment_Generation.zip?download=1"
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(DATA):
    !curl -L --fail --retry 5 -o {DATA} "{URL}"
with zipfile.ZipFile(DATA) as zf:
    for info in zf.infolist():
        print(info.filename, info.file_size)

## 6. Dry run

200 rows through every stage with the real model: train, generate (base and tuned), GGUF export.
It takes about 15 minutes, most of it building llama.cpp for the export. Check the `stage ... done
in` lines: they are what the real run's time is estimated from. The metrics mean nothing.

In [ ]:
dry = "--head --train-rows 200 --valid-rows 20 --test-rows 20"
dry += " --eval-steps 10 --logging-steps 5 --max-new-tokens 64"
!CUDA_VISIBLE_DEVICES=0 python -m training.sft.run --data {DATA} --out {WORK}/dry {dry}

## 7. Real run

Run it only after the dry run passed. Progress is logged in the cell output.

In [ ]:
args = f"--data {DATA} --out {OUT} --train-rows {TRAIN_ROWS}"
args += f" --scratch {WORK}/export"
!CUDA_VISIBLE_DEVICES=0 python -m training.sft.run {args}

## 8. Tidy the outputs

Keeps the quantized GGUF and removes the full-precision one the export makes on the way (about
6 GB), so the saved output stays small.

In [ ]:
import glob

for path in glob.glob(f"{OUT}/gguf/**/*.gguf", recursive=True):
    name = os.path.basename(path).lower()
    if "q4_k_m" not in name:
        print("removing", path)
        os.remove(path)
for root, _, files in os.walk(OUT):
    for name in files:
        path = os.path.join(root, name)
        print(path.replace(OUT, ""), os.path.getsize(path))

## 9. Results

Prints what goes into `eval/reports/sft.md` and the model card.

In [ ]:
import json

with open(f"{OUT}/metrics.json") as fh:
    m = json.load(fh)
print(json.dumps({k: m.get(k) for k in ("stages", "test", "eval_loss", "gguf")}, indent=2))